# Hospital patient data cleaning and wrangling



The file contains patient-level identifiers and encounter-like records. A repeated `patient_id` is not automatically a duplicate encounter: the repeated IDs in this file have different operational values. Exact duplicate rows are treated separately. The notebook does not infer or repair clinical facts from statistical outliers.

## 1. Set up and load the source









In [ ]:
from pathlib import Path
import pandas as pd
from pandas.api.types import is_string_dtype
from IPython.display import display

SOURCE_FILE = Path.cwd() / "clean_patients.csv"
OUTPUT_FILE = Path.cwd() / "clean_patients_cleaned.csv"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {SOURCE_FILE}. Open/run this notebook with the project folder as the working directory."
    )

raw = pd.read_csv(SOURCE_FILE, low_memory=False)
print(f"Source: {SOURCE_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
raw.head()

## 2. Profile missing values, duplicates, and types

**Why:** We need a baseline before editing so that each change can be measured. Missing values, fully repeated rows, and repeated patient IDs mean different things. A `patient_id` may identify a person across multiple visits, so it is not assumed to be a unique row key.

In [ ]:
print("Column types:")
print(raw.dtypes.to_string())
print("\nMissing values by column:")
print(raw.isna().sum().to_string())
print(f"\nExact duplicate rows: {raw.duplicated().sum():,}")
print(f"Rows with a repeated patient_id: {raw['patient_id'].duplicated(keep=False).sum():,}")
print(f"Distinct patient_id values: {raw['patient_id'].nunique(dropna=False):,}")
print("\nCategorical value counts (up to 20 per column):")
for column in raw.columns:
    if is_string_dtype(raw[column]):
        print(f"\n{column} ({raw[column].nunique(dropna=False)} distinct)")
        print(raw[column].value_counts(dropna=False).head(20).to_string())

## 3. Normalize text and make data types explicit

**Why:** Leading/trailing whitespace and blank strings can split one category into several labels or hide missing values. Trimming text is safe because whitespace is not part of these labels or IDs. Dates and numeric measures should have the types needed for comparisons and analysis. This step reports how many text values changed.

In [3]:
df = raw.copy()
text_columns = [column for column in df.columns if is_string_dtype(df[column])]
text_values_changed = 0
for column in text_columns:
    before = df[column].copy()
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace("", pd.NA)
    text_values_changed += int((before.fillna("<NA>") != df[column].fillna("<NA>")).sum())

numeric_columns = [
    "age", "treatment_cost", "opd_waiting_time_min",
    "room_allocation_time_min", "patient_rating"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"Text values trimmed/blank-normalized: {text_values_changed:,}")
print("Numeric columns converted:", ", ".join(numeric_columns))
print("Existing missing values after normalization:", int(df.isna().sum().sum()))

Text values trimmed/blank-normalized: 0
Numeric columns converted: age, treatment_cost, opd_waiting_time_min, room_allocation_time_min, patient_rating
Existing missing values after normalization: 0


## 4. Handle out-of-range ages

**Why:** Age `999` is a common placeholder-like value, and age `150` is beyond a deliberately generous 120-year upper validity bound. Neither should be silently treated as a real age in averages or age bands. Replace ages above 120 with missing (`NA`) so analysts can exclude or investigate them; do not guess a replacement. Age zero is retained because it can mean an infant under one year when age is recorded in completed years. This is a documented data rule, not a clinical diagnosis.

In [4]:
MAX_VALID_AGE = 120
invalid_age_mask = df["age"].notna() & ~df["age"].between(0, MAX_VALID_AGE)
invalid_age_count = int(invalid_age_mask.sum())
invalid_age_values = df.loc[invalid_age_mask, "age"].value_counts().sort_index()
print(f"Ages outside 0–{MAX_VALID_AGE}: {invalid_age_count:,}")
print("Values to set missing:")
print(invalid_age_values.to_string())
df.loc[invalid_age_mask, "age"] = pd.NA
df["age"] = df["age"].astype("Int64")

Ages outside 0–120: 85
Values to set missing:
age
150    46
999    39


## 5. Remove exact duplicates, impute missing values, retain repeated patients

**Why:** Four rows match every recorded field exactly, so they add no distinct information and can inflate row-based counts. Remove one copy of each exact row. Keep records that share a `patient_id` but differ in waiting time, room allocation time, rating, or another field: the dataset has no encounter ID, and those differences may represent separate recorded encounters. For future production use, add a unique encounter/visit key at the source.

**Missing-value rule:** For numeric fields with missing values, use the mean when the observed distribution is approximately symmetric (`|skewness| <= 0.5`) and the median when it is skewed. Round the age mean to a whole year. For categorical fields, use the mode. Do not infer IDs or dates. Here, the 85 out-of-range ages were set missing; valid ages are approximately symmetric (skewness about -0.107; mean about 43.33; median 44), so those ages are imputed with 43. No categorical, ID, or date values are missing.

In [5]:
exact_duplicates = int(df.duplicated(keep="first").sum())
df = df.drop_duplicates(keep="first").copy()
print(f"Exact duplicate rows removed: {exact_duplicates:,}")
print(f"Rows remaining: {len(df):,}")
print(f"Patient IDs still appearing more than once: {int(df['patient_id'].duplicated(keep=False).sum()):,}")

numeric_imputation_counts = {}
for column in numeric_columns:
    missing_count = int(df[column].isna().sum())
    if missing_count == 0:
        continue

    observed = df[column].dropna()
    if observed.empty:
        raise ValueError(f"Cannot impute {column}: no observed values are available")
    skewness = float(observed.skew()) if len(observed) > 2 else 0.0
    if abs(skewness) <= 0.5:
        fill_value = observed.mean()
        method = "mean"
    else:
        fill_value = observed.median()
        method = "median"
    if column == "age":
        fill_value = int(round(fill_value))

    df[column] = df[column].fillna(fill_value)
    numeric_imputation_counts[column] = missing_count
    print(f"{column}: imputed {missing_count:,} missing values with {method} = {fill_value} (skewness={skewness:.3f})")

categorical_imputation_counts = {}
non_imputable_text_columns = {"patient_id", "doctor_id", "admission_date", "discharge_date"}
categorical_columns = [column for column in text_columns if column not in non_imputable_text_columns]
for column in categorical_columns:
    missing_count = int(df[column].isna().sum())
    if missing_count == 0:
        continue

    modes = df[column].mode(dropna=True)
    if modes.empty:
        raise ValueError(f"Cannot impute {column}: no observed categories are available")
    df[column] = df[column].fillna(modes.iloc[0])
    categorical_imputation_counts[column] = missing_count
    print(f"{column}: imputed {missing_count:,} missing values with mode = {modes.iloc[0]}")

Exact duplicate rows removed: 4
Rows remaining: 40,596
Patient IDs still appearing more than once: 1,192
age: imputed 85 missing values with mean = 43 (skewness=-0.107)


## 6. Parse dates and audit numeric/date ranges

**Why:** Parsed dates support chronological comparisons and length-of-stay calculations. Invalid dates become missing and are counted, rather than guessed. Range checks help find impossible values; high treatment costs alone are not changed because expensive care can be legitimate and there is no external billing reference in this file.

In [6]:
date_columns = ["admission_date", "discharge_date"]
for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

range_checks = {
    "nonpositive treatment cost": df["treatment_cost"] <= 0,
    "negative OPD waiting time": df["opd_waiting_time_min"] < 0,
    "negative room allocation time": df["room_allocation_time_min"] < 0,
    "rating outside 1–5": ~df["patient_rating"].between(1, 5),
    "unparseable admission date": df["admission_date"].isna(),
    "unparseable discharge date": df["discharge_date"].isna(),
    "discharge before admission": df["discharge_date"] < df["admission_date"],
}
for label, mask in range_checks.items():
    print(f"{label}: {int(mask.fillna(False).sum()):,}")

print("\nNumeric summary after missing-value imputation:")
display(df[numeric_columns].describe().T)

nonpositive treatment cost: 0
negative OPD waiting time: 0
negative room allocation time: 0
rating outside 1–5: 0
unparseable admission date: 0
unparseable discharge date: 0
discharge before admission: 0

Numeric summary after missing-value imputation:


,count,mean,std,min,25%,50%,75%,max
age,40596.0,43.327397,20.356593,0.0,28.0,44.0,59.0,85.0
treatment_cost,40596.0,27121.623066,56738.755009,500.0,2505.0,5076.0,7914.0,376977.0
opd_waiting_time_min,40596.0,33.722953,29.803534,0.0,5.0,27.0,57.0,120.0
room_allocation_time_min,40596.0,8.427037,18.942579,0.0,0.0,0.0,0.0,90.0
patient_rating,40596.0,3.956991,1.037882,1.0,3.0,4.0,5.0,5.0


## 7. Check relationships between fields

**Why:** Some values are only meaningful for one service type. In this file, OPD rows should have a waiting time and no room-allocation time, while IPD rows should have room-allocation time and no OPD waiting time. These are consistency checks, not automatic correction rules. Any exceptions should be reviewed with the data owner before changing them. Emergency/referral relationships are also checked because the labels appear tightly linked in this extract.

In [7]:
opd = df["patient_type"].eq("OPD")
ipd = df["patient_type"].eq("IPD")
relationship_checks = {
    "OPD row with zero waiting time": opd & df["opd_waiting_time_min"].eq(0),
    "OPD row with room-allocation time": opd & df["room_allocation_time_min"].ne(0),
    "IPD row with nonzero OPD waiting time": ipd & df["opd_waiting_time_min"].ne(0),
    "IPD row with zero room-allocation time": ipd & df["room_allocation_time_min"].eq(0),
    "Emergency category not IPD": df["patient_category"].eq("Emergency") & ~ipd,
    "Emergency category without Emergency referral": df["patient_category"].eq("Emergency") & ~df["referral_source"].eq("Emergency"),
}
for label, mask in relationship_checks.items():
    print(f"{label}: {int(mask.fillna(False).sum()):,}")

print("\nService type by patient category:")
display(pd.crosstab(df["patient_category"], df["patient_type"], margins=True))

OPD row with zero waiting time: 0
OPD row with room-allocation time: 0
IPD row with nonzero OPD waiting time: 0
IPD row with zero room-allocation time: 0
Emergency category not IPD: 0
Emergency category without Emergency referral: 0

Service type by patient category:


patient_type,IPD,OPD,All
patient_category,,,
Appointment,3756,10448,14204
Emergency,6081,0,6081
Walk-in,0,20311,20311
All,9837,30759,40596


## 8. Validate and export

**Why:** A final check confirms that the output has no exact duplicate rows or missing values, all ages satisfy the chosen bound, and dates are ordered. The cleaned CSV is a derived file; the original remains untouched. Dates are serialized in ISO format by pandas.

In [8]:
assert df.duplicated().sum() == 0, "Exact duplicate rows remain"
assert df["age"].dropna().between(0, MAX_VALID_AGE).all(), "Age bound check failed"
assert int(df.isna().sum().sum()) == 0, "Missing values remain after imputation"
assert not (df["discharge_date"] < df["admission_date"]).fillna(False).any(), "Date ordering check failed"

df.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m-%d")

summary = pd.DataFrame({
    "measure": [
        "source rows", "cleaned rows", "exact duplicate rows removed",
        "distinct patient IDs", "rows with repeated patient ID",
        "invalid ages flagged", "age values imputed",
        "numeric values imputed", "categorical values imputed",
        "missing values in cleaned data"
    ],
    "value": [
        len(raw), len(df), exact_duplicates, df["patient_id"].nunique(dropna=False),
        int(df["patient_id"].duplicated(keep=False).sum()), invalid_age_count,
        numeric_imputation_counts.get("age", 0), sum(numeric_imputation_counts.values()),
        sum(categorical_imputation_counts.values()),
        int(df.isna().sum().sum())
    ]
})
display(summary)
print(f"\nSaved cleaned data to: {OUTPUT_FILE}")

,measure,value
0,source rows,40600
1,cleaned rows,40596
2,exact duplicate rows removed,4
3,distinct patient IDs,40000
4,rows with repeated patient ID,1192
5,invalid ages flagged,85
6,age values imputed,85
7,numeric values imputed,85
8,categorical values imputed,0
9,missing values in cleaned data,0



Saved cleaned data to: F:\Hospital patient analytics dashboard\clean_patients_cleaned.csv


## Cleaning decisions at a glance

- **Applied:** trim text fields and convert blanks to missing; parse date and numeric fields; set ages above 120 to missing; remove exact duplicate rows; impute numeric missing values with the mean for approximately symmetric distributions and the median for skewed distributions; impute categorical missing values with the mode.
- **Preserved:** repeated patient IDs with differing row details, age zero, all recorded categories and ratings, and high treatment costs.
- **Needs source-owner review if encountered on a future refresh:** invalid dates or numeric values and service-type timing inconsistencies. The current file's date order and service timing checks pass.
- **Modeling note:** this output is row-level data. Use distinct `patient_id` for patient counts, and use a true encounter ID for visit counts when one becomes available.